In [ ]:
import numpy as np
import bacco
import matplotlib.pyplot as plt

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
def plot_comparison(zoom, mtng, indices, final_sel):
    M1 = 1e10 * zoom.fof['halo_m200b'][indices['ind']]
    M2 = 1e10 * mtng.fof['halo_m200b'][final_sel]

    pos1 = zoom.fof['halo_pos'][indices['ind']]
    pos2 = mtng.fof['halo_pos'][final_sel].T

    mfof = zoom.fof['halo_mfof_type']
    hd = mfof[:,0] + mfof[:,1] + mfof[:,4] + mfof[:,5]
    lr = mfof[:,2] + mfof[:,3]
    f_all = lr / (hd + lr)         # shape (N_zoom_halos,)
    f_contam = f_all[indices['ind']]           # shape (N_targets, k), matches d_m

    fig, ax = plt.subplots(2, 2, dpi=200, figsize=(11,10))

    ax[0,0].set_xscale('log')
    ax[0,1].set_xscale('log')
    ax[1,0].set_xscale('log')
    ax[1,1].set_xscale('log')
    ax[1,1].set_yscale('log')

    ax[0,0].plot(M2, M1/M2, ls='', marker='o')
    ax[0,1].plot(M2, indices['d'], ls='', marker='o')
    ax[1,0].plot(M2, f_contam, ls='', marker='o')
    ax[1,1].plot(M2, M1, ls='', marker='o')

    return 0

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264
zoom = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap), numpart=4320**3, use_ids=True)

print(zoom.header['Redshift'])

In [ ]:
zoom.fof['halo_vel']

In [ ]:
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=snap)
mtng.fof['halo_pos'][:,0] = (mtng.fof['halo_pos'][:,0] - 125) % 500
mtng.sub['pos'][:,0] = (mtng.sub['pos'][:,0] - 125) % 500

print(mtng.header['Redshift'])

In [ ]:
with open("/cosmos_storage/home/fgmaion/MTNG-resims/halo_selection/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

In [ ]:
ind = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/halo_selection/cross_match_fiducial.npy", allow_pickle=True).item()

In [ ]:
M1 = 1e10 * zoom.fof['halo_m200b'][ind['ind']]
M2 = 1e10 * mtng.fof['halo_m200b'][final_sel]

pos1 = zoom.fof['halo_pos'][ind['ind']]
pos2 = mtng.fof['halo_pos'][final_sel].T

In [ ]:
plot_comparison(zoom, mtng, ind, final_sel)

In [ ]:
xmatch = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/cross-match/cross_match_fiducial.npy", allow_pickle=True).item()

In [ ]:
xmatch_2 = utils.cross_match(zoom, snap=264, name=None)

In [ ]:
plot_comparison(zoom, mtng, xmatch_2, final_sel)